# PM2.5 Forecasting in Indian Cities: Data Pipeline, MLP Baseline and 1D-CNN
ICT-4442 Deep Learning Mini Project

Runtime: GPU (Runtime > Change runtime type > GPU). Upload `city_day.csv` when prompted.

## Setup
The MLP is the baseline model. Set `INCLUDE_REFERENCE = False` to remove the naive reference rows (persistence and 7-day average).

In [1]:
import os, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display

WINDOW = 7
HORIZONS = [1, 3, 7]
HMAX = max(HORIZONS)
MAX_FILL = 3
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
SEEDS = [0, 1, 2]
BASELINE = "MLP"
INCLUDE_REFERENCE = True
POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "CO", "SO2", "O3"]
DROP_CITIES = ["Mumbai", "Jorapokhar"]
DROP_COLS = ["AQI", "AQI_Bucket", "Xylene", "Toluene", "Benzene", "NH3"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. In Colab: Runtime > Change runtime type > select a GPU, then restart and run all.")
print("Device:", DEVICE)

GPU: Tesla T4
Device: cuda


## Load data

In [2]:
if not os.path.exists("city_day.csv"):
    from google.colab import files
    files.upload()

raw = pd.read_csv("city_day.csv", parse_dates=["Date"])
print("Raw shape:", raw.shape)
print("Date range:", raw["Date"].min().date(), "to", raw["Date"].max().date())

Saving city_day.csv to city_day.csv
Raw shape: (29531, 16)
Date range: 2015-01-01 to 2020-07-01


## Phase 1: Filter, sort and daily reindex

In [3]:
df = (raw[~raw["City"].isin(DROP_CITIES)]
      .drop(columns=DROP_COLS)
      .sort_values(["City", "Date"])
      .reset_index(drop=True))

frames = []
for city, g in df.groupby("City"):
    idx = pd.date_range(g["Date"].min(), g["Date"].max(), freq="D")
    g = g.set_index("Date")[POLLUTANTS].reindex(idx)
    g.index.name = "Date"
    g["City"] = city
    frames.append(g.reset_index())
daily = pd.concat(frames, ignore_index=True)

print("Cities:", daily["City"].nunique())
print("Rows after filtering:", len(df), "| rows after daily reindex:", len(daily))
print("PM2.5 missing (%):", round(daily["PM2.5"].isna().mean() * 100, 1))

Cities: 24
Rows after filtering: 26353 | rows after daily reindex: 26353
PM2.5 missing (%): 9.8


## Phase 1: Error handling
Rows where PM2.5 exceeds 1.5 times PM10, isolated PM2.5 spikes and PM10 sensor-cap values are set to missing. Smaller PM2.5 > PM10 cases are kept and counted.

In [4]:
both = daily["PM2.5"].notna() & daily["PM10"].notna()
pm25_gt_pm10 = both & (daily["PM2.5"] > daily["PM10"])
invalid_pair = both & (daily["PM2.5"] > 1.5 * daily["PM10"])
local_median = daily.groupby("City")["PM2.5"].transform(
    lambda s: s.rolling(7, center=True, min_periods=3).median())
spike = (daily["PM2.5"] > 300) & (daily["PM2.5"] > 5 * local_median)
pm10_cap = daily["PM10"] >= 1000

cleaning_log = {
    "PM2.5 > PM10 (all)": int(pm25_gt_pm10.sum()),
    "PM2.5 > 1.5 x PM10 (set PM2.5 and PM10 to NaN)": int(invalid_pair.sum()),
    "Isolated PM2.5 spike >300 and >5x local median (set PM2.5 and PM10 to NaN)": int(spike.sum()),
    "PM10 at sensor cap >=1000 (set PM10 to NaN)": int(pm10_cap.sum()),
    "PM2.5 > PM10 with ratio <=1.5 (kept)": int((pm25_gt_pm10 & ~invalid_pair).sum()),
}

invalid = invalid_pair | spike
daily.loc[invalid, ["PM2.5", "PM10"]] = np.nan
daily.loc[pm10_cap, "PM10"] = np.nan

for k, v in cleaning_log.items():
    print(f"{v:6d}  {k}")
print("Rows with PM2.5 invalidated:", int(invalid.sum()))

   247  PM2.5 > PM10 (all)
   111  PM2.5 > 1.5 x PM10 (set PM2.5 and PM10 to NaN)
    19  Isolated PM2.5 spike >300 and >5x local median (set PM2.5 and PM10 to NaN)
     1  PM10 at sensor cap >=1000 (set PM10 to NaN)
   136  PM2.5 > PM10 with ratio <=1.5 (kept)
Rows with PM2.5 invalidated: 126


## Phase 1: Short-gap fill
Gaps of up to 3 days are forward filled. Longer gaps are left missing.

In [5]:
daily["PM2.5_obs"] = daily["PM2.5"]

def fill_short_gaps(s, limit=MAX_FILL):
    isn = s.isna()
    stretch = (isn != isn.shift()).cumsum()
    length = isn.groupby(stretch).transform("size")
    fillable = isn & (length <= limit)
    return s.where(~fillable, s.ffill())

for col in POLLUTANTS:
    daily[col] = daily.groupby("City")[col].transform(fill_short_gaps)

daily["PM10_missing"] = daily["PM10"].isna().astype(np.float32)
print("Missing after short-gap fill (%):")
print((daily[POLLUTANTS].isna().mean() * 100).round(1).to_string())

Missing after short-gap fill (%):
PM2.5     9.0
PM10     35.8
NO        6.9
NO2       6.9
NOx       8.8
CO        4.7
SO2       7.6
O3        8.5


## Phase 2: Chronological split, imputation and scaling
Per-city split by time (70 / 15 / 15). Median fill, log transform and standardization use training rows only.

In [6]:
daily["row"] = daily.groupby("City").cumcount()
daily["n"] = daily.groupby("City")["row"].transform("size")
train_end = (daily["n"] * TRAIN_FRAC).astype(int)
val_end = (daily["n"] * (TRAIN_FRAC + VAL_FRAC)).astype(int)
daily["period"] = np.select([daily["row"] < train_end, daily["row"] < val_end], [0, 1], 2)

train_rows = daily["period"] == 0
train_median = daily.loc[train_rows, POLLUTANTS].median()

feat = np.log1p(daily[POLLUTANTS].fillna(train_median))
mu = feat[train_rows].mean()
sd = feat[train_rows].std()
scaled = (feat - mu) / sd

month = daily["Date"].dt.month
scaled["PM10_missing"] = daily["PM10_missing"]
scaled["month_sin"] = np.sin(2 * np.pi * month / 12)
scaled["month_cos"] = np.cos(2 * np.pi * month / 12)
CHANNELS = list(scaled.columns)
print("Channels:", CHANNELS)

Channels: ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3', 'PM10_missing', 'month_sin', 'month_cos']


## Phase 2: Build 7-day windows
Targets: PM2.5 at t+1, t+3 and t+7, stored as change from today. Windows are kept only if today and all three targets were measured.

In [7]:
cities = sorted(daily["City"].unique())
city_to_id = {c: i for i, c in enumerate(cities)}
N_CITIES = len(cities)

X_list, y_list, base_list, pm_win_list = [], [], [], []
cid_list, date_list, split_list = [], [], []

for city, g in daily.groupby("City"):
    idx = g.index
    vals = scaled.loc[idx, CHANNELS].to_numpy(np.float32)
    obs = g["PM2.5_obs"].to_numpy()
    filled = g["PM2.5"].to_numpy()
    period = g["period"].to_numpy()
    dates = g["Date"].to_numpy()
    for i in range(WINDOW - 1, len(g) - HMAX):
        targets = [obs[i + h] for h in HORIZONS]
        if np.isnan(obs[i]) or np.isnan(targets).any():
            continue
        X_list.append(vals[i - WINDOW + 1:i + 1])
        y_list.append(targets)
        base_list.append(obs[i])
        pm_win_list.append(filled[i - WINDOW + 1:i + 1])
        cid_list.append(city_to_id[city])
        date_list.append(dates[i])
        split_list.append(period[i + HMAX])

X = np.stack(X_list).astype(np.float32)
y_abs = np.array(y_list, dtype=np.float32)
base = np.array(base_list, dtype=np.float32)
pm_win = np.stack(pm_win_list).astype(np.float32)
city_id = np.array(cid_list)
win_date = np.array(date_list)
split = np.array(split_list)

y_res = y_abs - base[:, None]
tr, va, te = split == 0, split == 1, split == 2
y_scale = y_res[tr].std(axis=0)

print("X:", X.shape, "| y:", y_abs.shape)
print("Windows train/val/test:", tr.sum(), va.sum(), te.sum())
print("Residual scale (train std) per horizon:", y_scale.round(1))

summary = (pd.DataFrame({"city": [cities[i] for i in city_id], "split": split})
           .pivot_table(index="city", columns="split", aggfunc="size", fill_value=0)
           .rename(columns={0: "train", 1: "val", 2: "test"}))
display(summary.sort_values("test"))

X: (22244, 7, 11) | y: (22244, 3)
Windows train/val/test: 14753 3756 3735
Residual scale (train std) per horizon: [33.6 44.5 49.7]


split,train,val,test
city,,,
Aizawl,66,16,11
Ernakulam,96,24,18
Kochi,100,24,25
Shillong,115,14,43
Bhopal,181,35,44
Chandigarh,184,46,46
Coimbatore,243,48,52
Guwahati,332,73,76
Brajrajnagar,446,125,77


## Phase 3: Naive reference methods (no training)
Persistence and 7-day average, evaluated on the same test windows. These are extra reference rows, not the baseline model.

In [8]:
def metrics(y_true, y_pred):
    err = (y_pred - y_true).astype(np.float64)
    return np.sqrt((err ** 2).mean(axis=0)), np.abs(err).mean(axis=0)

reference = {}
if INCLUDE_REFERENCE:
    persist = np.repeat(base[:, None], len(HORIZONS), axis=1)
    avg7 = np.repeat(np.nanmean(pm_win, axis=1)[:, None], len(HORIZONS), axis=1)
    zeros = np.zeros(len(HORIZONS))
    for name, pred in [("Naive: persistence", persist), ("Naive: 7-day average", avg7)]:
        r, m = metrics(y_abs[te], pred[te])
        reference[name] = {"rmse": r, "rmse_sd": zeros, "mae": m, "mae_sd": zeros}
    ref_df = pd.DataFrame(
        {name: np.concatenate([v["rmse"], v["mae"]]) for name, v in reference.items()},
        index=[f"RMSE t+{h}" for h in HORIZONS] + [f"MAE t+{h}" for h in HORIZONS]).round(2)
    display(ref_df)

,Naive: persistence,Naive: 7-day average
RMSE t+1,23.14,29.76
RMSE t+3,36.25,34.19
RMSE t+7,41.83,35.95
MAE t+1,12.56,16.52
MAE t+3,19.91,18.99
MAE t+7,22.70,20.45


## Phase 4: Model definitions

In [9]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(192, 96), out_dim=3, dropout=0.1):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x, city):
        return self.net(x)


class CNN1D(nn.Module):
    def __init__(self, in_ch, n_cities, window=WINDOW, emb_dim=8, channels=32,
                 hidden=128, out_dim=3, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(n_cities, emb_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, channels, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1), nn.ReLU())
        self.head = nn.Sequential(
            nn.Linear(channels * window + emb_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, out_dim))

    def forward(self, x, city):
        z = self.conv(x).flatten(1)
        return self.head(torch.cat([z, self.emb(city)], dim=1))


def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Phase 4: Inputs and training utilities

In [10]:
onehot = np.eye(N_CITIES, dtype=np.float32)[city_id]
n_pollutant_ch = len(POLLUTANTS) + 1
X_mlp = np.concatenate([X[:, :, :n_pollutant_ch].reshape(len(X), -1), X[:, -1, n_pollutant_ch:], onehot], axis=1)
X_cnn = np.transpose(X, (0, 2, 1)).copy()

city_t = torch.tensor(city_id, dtype=torch.long, device=DEVICE)
y_t = torch.tensor(y_res / y_scale, dtype=torch.float32, device=DEVICE)
idx_tr, idx_va, idx_te = [torch.tensor(np.where(m)[0], device=DEVICE) for m in (tr, va, te)]

def fit(model, x, epochs=150, patience=15, lr=1e-3, weight_decay=1e-5, batch_size=256):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    best, best_state, wait = float("inf"), None, 0
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        model.train()
        perm = idx_tr[torch.randperm(len(idx_tr), device=DEVICE)]
        total = 0.0
        for start in range(0, len(perm), batch_size):
            b = perm[start:start + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(x[b], city_t[b]), y_t[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(x[idx_va], city_t[idx_va]), y_t[idx_va]).item()
        history["train"].append(total / len(perm))
        history["val"].append(val_loss)
        if val_loss < best - 1e-5:
            best, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, history

def predict_test(model, x):
    model.eval()
    with torch.no_grad():
        out = model(x[idx_te], city_t[idx_te]).cpu().numpy()
    return np.clip(base[te][:, None] + out * y_scale, 0, None)

## Phase 4: Train and evaluate MLP (baseline) and 1D-CNN
Three seeds per model, early stopping on validation loss.

In [11]:
results = {}
histories = {}
test_predictions = {}

def run_model(name, build, x_np):
    x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    rm, ma, hist, preds = [], [], [], []
    for seed in SEEDS:
        set_seed(seed)
        model = build().to(DEVICE)
        model, h = fit(model, x)
        pred = predict_test(model, x)
        r, m = metrics(y_abs[te], pred)
        rm.append(r); ma.append(m); hist.append(h); preds.append(pred)
        print(f"{name} seed {seed}: epochs {len(h['val'])}, RMSE {np.round(r, 2)}, MAE {np.round(m, 2)}")
    params = n_params(build())
    results[name] = {"rmse": np.mean(rm, 0), "rmse_sd": np.std(rm, 0),
                     "mae": np.mean(ma, 0), "mae_sd": np.std(ma, 0)}
    histories[name] = hist
    test_predictions[name] = np.mean(preds, axis=0)
    print(f"{name}: {params} parameters\n")

run_model("MLP", lambda: MLP(X_mlp.shape[1]), X_mlp)
run_model("1D-CNN", lambda: CNN1D(X_cnn.shape[1], N_CITIES), X_cnn)

MLP seed 0: epochs 27, RMSE [21.73 30.84 33.41], MAE [12.28 17.96 20.07]
MLP seed 1: epochs 25, RMSE [21.92 31.36 33.35], MAE [12.31 18.21 20.1 ]
MLP seed 2: epochs 21, RMSE [22.05 31.07 33.32], MAE [12.19 17.72 19.67]
MLP: 36099 parameters

1D-CNN seed 0: epochs 27, RMSE [22.4  31.34 33.76], MAE [12.49 17.96 20.3 ]
1D-CNN seed 1: epochs 26, RMSE [21.92 31.38 34.7 ], MAE [12.22 18.26 20.59]
1D-CNN seed 2: epochs 26, RMSE [21.81 30.97 33.91], MAE [12.18 18.29 20.63]
1D-CNN: 34595 parameters



## Results
Main comparison is against the MLP baseline. Naive reference rows appear at the bottom of the table.

In [12]:
def fmt(mean, sd):
    return f"{mean:.2f} ± {sd:.2f}" if sd > 0 else f"{mean:.2f}"

rows = {**results, **reference}
labels = {BASELINE: f"{BASELINE} (baseline)"}
cols = pd.MultiIndex.from_product([[f"t+{h}" for h in HORIZONS], ["RMSE", "MAE"]])
table = pd.DataFrame(index=[labels.get(n, n) for n in rows], columns=cols)
for name, v in rows.items():
    for j, h in enumerate(HORIZONS):
        table.loc[labels.get(name, name), (f"t+{h}", "RMSE")] = fmt(v["rmse"][j], v["rmse_sd"][j])
        table.loc[labels.get(name, name), (f"t+{h}", "MAE")] = fmt(v["mae"][j], v["mae_sd"][j])
display(table)

others = [n for n in results if n != BASELINE]
vs_baseline = pd.DataFrame(
    {n: (results[BASELINE]["rmse"] - results[n]["rmse"]) / results[BASELINE]["rmse"] * 100 for n in others},
    index=[f"t+{h}" for h in HORIZONS]).round(1)
print(f"RMSE change vs {BASELINE} baseline (%, positive = better than {BASELINE}):")
display(vs_baseline)

if INCLUDE_REFERENCE:
    best_ref = np.minimum(reference["Naive: persistence"]["rmse"], reference["Naive: 7-day average"]["rmse"])
    gain = pd.DataFrame({n: (best_ref - results[n]["rmse"]) / best_ref * 100 for n in results},
                        index=[f"t+{h}" for h in HORIZONS]).round(1)
    print("RMSE gain over the better naive reference method (%):")
    display(gain)

t+1                         t+3                \
                              RMSE           MAE          RMSE           MAE   
MLP (baseline)        21.90 ± 0.13  12.26 ± 0.05  31.09 ± 0.21  17.96 ± 0.20   
1D-CNN                22.04 ± 0.26  12.30 ± 0.14  31.23 ± 0.18  18.17 ± 0.15   
Naive: persistence           23.14         12.56         36.25         19.91   
Naive: 7-day average         29.76         16.52         34.19         18.99   

                               t+7                
                              RMSE           MAE  
MLP (baseline)        33.36 ± 0.04  19.95 ± 0.19  
1D-CNN                34.12 ± 0.41  20.51 ± 0.15  
Naive: persistence           41.83         22.70  
Naive: 7-day average         35.95         20.45

RMSE change vs MLP baseline (%, positive = better than MLP):


,1D-CNN
t+1,-0.7
t+3,-0.4
t+7,-2.3


RMSE gain over the better naive reference method (%):


,MLP,1D-CNN
t+1,5.4,4.7
t+3,9.1,8.7
t+7,7.2,5.1


## Save artifacts
Share `pm25_windows.npz` with the team so all four models use identical windows and splits.

In [13]:
np.savez_compressed("pm25_windows.npz", X=X, y_abs=y_abs, base=base, pm_win=pm_win,
                    city_id=city_id, split=split, win_date=win_date.astype("datetime64[D]").astype(str),
                    y_scale=y_scale, cities=np.array(cities), channels=np.array(CHANNELS))
table.to_csv("results_table.csv")
print("Saved pm25_windows.npz and results_table.csv")

Saved pm25_windows.npz and results_table.csv
